# KADMON — Comparaison des techniques OT

**Kantorovich Anatomical Deviation Mapping of Neurotracts**

Ce notebook compare les six configurations compression × transport sur une seule paire de bundles. QuickBundles sert de baseline tractographique pour les méthodes K-Means et binning. Utilisez ensuite `2_Visualiser_Deplacements_3D.ipynb` pour explorer la méthode choisie dans FURY.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"

SOURCE_PATH = BUNDLES_DIR / "103818/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"
TARGET_PATH = BUNDLES_DIR / "433839/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"

In [2]:
import pandas as pd

from kadmon.comparison import ComparisonConfiguration, compare_configurations
from kadmon.io import load_bundle

from kadmon.defaults import (
    BINNING_PARTIAL_BIN_SIZE,
    BINNING_PARTIAL_BINNING_NB,
    BINNING_PARTIAL_MASS,
    BINNING_PARTIAL_REPRESENTATIVE,
    BINNING_SINKHORN_BIN_SIZE,
    BINNING_SINKHORN_BINNING_NB,
    BINNING_SINKHORN_EPSILON,
    BINNING_SINKHORN_REPRESENTATIVE,
    KMEANS_PARTIAL_K,
    KMEANS_PARTIAL_MASS,
    KMEANS_SINKHORN_EPSILON,
    KMEANS_SINKHORN_K,
    QUICKBUNDLES_PARTIAL_MASS,
    QUICKBUNDLES_PARTIAL_THRESHOLD,
    QUICKBUNDLES_SINKHORN_EPSILON,
    QUICKBUNDLES_SINKHORN_THRESHOLD,
)

N_POINTS = 12
SEED = 42
MAX_REPRESENTATIVES = 5000


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## Chargement

In [3]:
if not SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Bundle source introuvable : {SOURCE_PATH}")
if not TARGET_PATH.is_file():
    raise FileNotFoundError(f"Bundle cible introuvable : {TARGET_PATH}")

bundle_a = load_bundle(SOURCE_PATH, n_points=N_POINTS)
bundle_b = load_bundle(TARGET_PATH, n_points=N_POINTS)
print(f"Source : {len(bundle_a)} streamlines")
print(f"Cible  : {len(bundle_b)} streamlines")

Source : 10954 streamlines
Cible  : 11497 streamlines


## Calcul des six techniques

In [4]:
sinkhorn = {"max_iter": 2000, "stop_threshold": 1e-6, "reject_threshold": 1e-5}
kmeans = {"max_iter": 300, "tol": 1e-4, "seed": SEED}

def configuration_from_parameters(name, parameters):
    compression, transport = name.rsplit("_", 1)
    if compression == "quickbundles":
        compression_parameters = {"threshold": parameters["threshold"]}
    elif compression == "kmeans":
        compression_parameters = {
            **kmeans,
            "n_clusters": min(parameters["n_clusters"], len(bundle_a), len(bundle_b)),
        }
    else:
        compression_parameters = {
            "bin_size": parameters["bin_size"],
            "binning_nb": parameters["binning_nb"],
            "method": parameters["method"],
            "n_points": N_POINTS,
        }
    transport_parameters = (
        {"mass": parameters["mass"]}
        if transport == "partial"
        else {**sinkhorn, "epsilon": parameters["epsilon"]}
    )
    return ComparisonConfiguration(compression, transport, compression_parameters, transport_parameters)

default_parameters = {
    "quickbundles_partial": {"threshold": QUICKBUNDLES_PARTIAL_THRESHOLD, "mass": QUICKBUNDLES_PARTIAL_MASS},
    "quickbundles_sinkhorn": {"threshold": QUICKBUNDLES_SINKHORN_THRESHOLD, "epsilon": QUICKBUNDLES_SINKHORN_EPSILON},
    "kmeans_partial": {"n_clusters": KMEANS_PARTIAL_K, "mass": KMEANS_PARTIAL_MASS},
    "kmeans_sinkhorn": {"n_clusters": KMEANS_SINKHORN_K, "epsilon": KMEANS_SINKHORN_EPSILON},
    "binning_partial": {"bin_size": BINNING_PARTIAL_BIN_SIZE, "binning_nb": BINNING_PARTIAL_BINNING_NB, "method": BINNING_PARTIAL_REPRESENTATIVE, "mass": BINNING_PARTIAL_MASS},
    "binning_sinkhorn": {"bin_size": BINNING_SINKHORN_BIN_SIZE, "binning_nb": BINNING_SINKHORN_BINNING_NB, "method": BINNING_SINKHORN_REPRESENTATIVE, "epsilon": BINNING_SINKHORN_EPSILON},
}

anatomical_configurations = {
    name: configuration_from_parameters(name, parameters)
    for name, parameters in default_parameters.items()
}
anatomical_results = compare_configurations(
    bundle_a, bundle_b, anatomical_configurations,
    max_representatives=MAX_REPRESENTATIVES,
)


## 1. Profil anatomique retenu

In [5]:
anatomical_metrics = pd.DataFrame([
    result["metrics"] for result in anatomical_results.values()
])
display(anatomical_metrics.sort_values("configuration").reset_index(drop=True))


,configuration,compression,transport,source_n_representatives,target_n_representatives,source_compression_ratio,target_compression_ratio,compression_time_s,mdf_time_s,transport_time_s,global_distance_mm,transport_objective_normalized,transported_mass,mean_mm,median_mm,max_mm,p95_mm,n_valid_representatives,n_untransported_representatives
0,binning+partial,binning,partial,55,52,0.005021,0.004523,1.073708,0.000287,0.001068,4.379683,0.104675,0.68,3.659289,3.538123,7.065357,5.773711,40,15
1,binning+sinkhorn,binning,sinkhorn,407,406,0.037155,0.035314,1.098846,0.044064,3.814670,7.310855,0.290421,1.00,6.021180,5.362900,15.015970,10.337414,407,0
2,kmeans+partial,kmeans,partial,40,40,0.003652,0.003479,2.337414,0.000191,0.001116,3.669652,0.086070,0.63,3.189204,3.232054,6.879091,4.183224,27,13
3,kmeans+sinkhorn,kmeans,sinkhorn,155,155,0.014150,0.013482,2.953932,0.006570,0.829250,6.996959,0.300141,1.00,6.033846,5.168241,14.958808,12.345562,155,0
4,quickbundles+partial,quickbundles,partial,91,93,0.008307,0.008089,0.110580,0.000731,0.002116,6.592056,0.211002,0.99,6.028555,5.409592,14.655257,13.323899,91,0
5,quickbundles+sinkhorn,quickbundles,sinkhorn,160,173,0.014607,0.015047,0.171017,0.009914,0.058520,8.536750,0.238650,1.00,6.232998,5.430074,13.218651,11.046148,160,0


## 2. Synthèse des profils anatomiques

Le tableau rassemble les paramètres Optuna retenus, la RE-ID globale et les métriques calculées sur la paire courante pour les six profils anatomiques.


In [6]:
import optuna

STUDY_PATH = PROJECT_ROOT / "notebooks" / "optuna" / "studies" / "optuna_reid.sqlite3"
REID_TRIALS = {
    "quickbundles_partial": ("quickbundles_partial", 75),
    "quickbundles_sinkhorn": ("quickbundles_sinkhorn", 114),
    "kmeans_partial": ("kmeans_partial", 44),
    "kmeans_sinkhorn": ("kmeans_sinkhorn", 130),
    "binning_partial": ("binning_partial", 55),
    "binning_sinkhorn": ("binning_sinkhorn", 374),
}
TECHNIQUE_NAMES = {
    "quickbundles_partial": "QuickBundles + Partial OT",
    "quickbundles_sinkhorn": "QuickBundles + Sinkhorn",
    "kmeans_partial": "K-Means + Partial OT",
    "kmeans_sinkhorn": "K-Means + Sinkhorn",
    "binning_partial": "Binning + Partial OT",
    "binning_sinkhorn": "Binning + Sinkhorn",
}

def format_parameters(parameters):
    return "; ".join(
        f"{('k' if key == 'n_clusters' else 'representative' if key == 'method' else key)}={value:.8g}"
        if isinstance(value, float) else
        f"{('k' if key == 'n_clusters' else 'representative' if key == 'method' else key)}={value}"
        for key, value in parameters.items()
    )

def short_bundle_name(name):
    prefix = "tractosearch_nn_8_0mm_all_"
    return name.removeprefix(prefix).removesuffix("_m")

if not STUDY_PATH.is_file():
    raise FileNotFoundError(f"Base Optuna Re-ID introuvable : {STUDY_PATH}")
storage = f"sqlite:///{STUDY_PATH}"
optuna.logging.set_verbosity(optuna.logging.WARNING)

summary_rows = []
for key in TECHNIQUE_NAMES:
    study_name, trial_number = REID_TRIALS[key]
    study = optuna.load_study(study_name=study_name, storage=storage)
    trial = next((item for item in study.trials if item.number == trial_number), None)
    if trial is None or trial.state != optuna.trial.TrialState.COMPLETE:
        raise RuntimeError(f"Essai Re-ID indisponible : {study_name} #{trial_number}")
    reid = trial.user_attrs
    result_metrics = anatomical_results[key]["metrics"]
    total_time_s = sum(
        float(result_metrics[name])
        for name in ("compression_time_s", "mdf_time_s", "transport_time_s")
    )
    recorded_failures = reid.get("reid_failed_bundles")
    failures = (
        [short_bundle_name(name) for name in recorded_failures]
        if recorded_failures is not None else None
    )
    summary_rows.append({
        "Technique": TECHNIQUE_NAMES[key],
        "Trial Optuna": trial_number,
        "RE-ID score": f"{reid['intra_identity_top1_accuracy']:.1%}",
        "Bundles non Re-ID": (
            "Non enregistré" if failures is None
            else ", ".join(failures) if failures else "Aucun"
        ),
        "Paramètres": format_parameters(default_parameters[key]),
        "Déplacement moyen": f"{result_metrics['mean_mm']:.3f} mm",
        "Nombre de représentants": (
            f"{result_metrics['source_n_representatives']} source / "
            f"{result_metrics['target_n_representatives']} cible"
        ),
        "Temps d'exécution": f"{total_time_s:.3f} s",
    })

technique_summary = pd.DataFrame(summary_rows).set_index("Technique")
display(technique_summary)


,Trial Optuna,RE-ID score,Bundles non Re-ID,Paramètres,Déplacement moyen,Nombre de représentants,Temps d'exécution
Technique,,,,,,,
QuickBundles + Partial OT,75,93.5%,"CST_L, IFOF_R",threshold=7; mass=0.99,6.029 mm,91 source / 93 cible,0.113 s
QuickBundles + Sinkhorn,114,90.3%,"CST_L, ICP_L, IFOF_R",threshold=6; epsilon=0.087448171,6.233 mm,160 source / 173 cible,0.239 s
K-Means + Partial OT,44,100.0%,Non enregistré,k=40; mass=0.63,3.189 mm,40 source / 40 cible,2.339 s
K-Means + Sinkhorn,130,93.5%,"CST_L, IFOF_R",k=155; epsilon=0.017736981,6.034 mm,155 source / 155 cible,3.790 s
Binning + Partial OT,55,96.8%,CST_L,bin_size=16; binning_nb=2; representative=mean...,3.659 mm,55 source / 52 cible,1.075 s
Binning + Sinkhorn,374,96.8%,CST_L,bin_size=8; binning_nb=2; representative=mean;...,6.021 mm,407 source / 406 cible,4.958 s


## 3. Analyse des résultats

- **K-Means + Partial OT** obtient la meilleure RE-ID observée (**100 %**) dans ce protocole.
- **QuickBundles** est la compression la plus rapide sur la paire étudiée.
- **Binning + Sinkhorn** atteint **96,8 %** de RE-ID avec la représentation la plus fine des profils retenus.
- Les profils Optuna sont des points de départ anatomiques; ils peuvent nécessiter un nouvel ajustement sur d'autres données.
